## GTSEP v1 multi

### Description
Unconstrained batteries. Removed binary decision variables.

The model does not include degradation of assets, so all assets have an infinite lifetime and new investments are just added to existing ones. The key difference here with regard to implementation is two things:

1. Since the model is over multiple years, the objective function needs to consider the time value of money. I have solved this by calulcating the annualized investment costs to represent the capex costs and the opex costs are simply the cost of production. Both are discounted to the present value using the discount rate.
2. Since the model is over multiple years, basically all decision variables as well as input data has been extended with the year index y. I use the convention that for timesteps the year index comes first, then the timestep index (hour).

Difference from v1 multi is that this model uses a subset of weeks as representative periods. 

### Indexes and index sets

- $n \in N$: Set of nodes.
- $i \in G^{old}$: Set of existing generators (at node $n$).
- $i \in G^{new}$: Set of new generators (at node $n$).
- $i \in G$: Set of all generators.
- $i \in G_n$: Set of generators at node $n$, including new generators.
- $b \in B^{new}$: Set of new branches.
- $b \in B^{old}$: Set of existing branches.
- $b \in B$: Set of all branches.
- $b \in B_n^{in}$: Set of branches coming into node $n$, including new branches.
- $b \in B_n^{out}$: Set of branches going out of node $n$, including new branches.
- $s \in S^{old}$: Set of old batteries (at node $n$).
- $s \in S^{new}$: Set of new batteries (at node $n$).
- $s \in S$: Set of all batteries (at node $n$).
- $s \in S_n$: Set of batteries at node $n$, including new batteries.
- $y \in Y$: Set of years.
- $y' \in Y_y$: Set of years up to year $y$. $Y_y = \{y' \in Y \mid y' \leq y\}$.
- $w \in W$: Set of representative weeks.
- $t \in T_w$: Set of hourly periods within each representative week $w$.

### Parameters

- $P_{i}^{\min}$: Min power output generator $i$ (MW)
- $P_{i}^{\max}$: Max power output generator $i$ (MW)
- $VOLL$: Value of lost load (\$/MWh)
- $CC$: Cost of curtailment (\$/MWh)
- $MC_{i,y}$: Marginal cost of generator $i$ (\$/MWh)
- $CO2_{i,y}$: Cost of CO2 emissions generator $i$ (\$/MWh)
- $E_{i,y}$: CO2 emissions generator $i$ (ton/MWh)
- $E_{limit}$: CO2 emissions limit (ton)
- $D_{n,y,w,t}$: Demand at node $n$, year $y$, week $w$, hour $t$ (MW)
- $l_b$: Loss factor of branch $b$
- $P_{b,y}^{\max}$: Max power flow branch $b$ in year $y$ (MW)
- $\eta_{s}^{ch}$: Charge efficiency battery $s$
- $\eta_{s}^{dis}$: Discharge efficiency battery $s$
- $P_{s}^{ch,\max}, P_{s}^{ch,\min}$: Charging limits battery $s$ (MW)
- $P_{s}^{dis,\max}, P_{s}^{dis,\min}$: Discharging limits battery $s$ (MW)
- $SOC_{s}^{\max}, SOC_{s}^{\min}$: SOC limits battery $s$ (MWh)
- $MC_{s,y}^{dis}$: Marginal discharge cost battery $s$ (\$/MWh)
- $cf_{i,y,w,t}$: Capacity factor of generator $i$ at year $y$, week $w$, hour $t$
- $P_b^{min}, P_b^{\max}$: Min/max capacity of new branch $b$ (MW)
- $AIC_{i,y}, AIC_{s,y}, AIC_{b,y}$: Annualized investment costs generators, batteries, branches (\$/MW or \$/MWh)
- $weight_w$: Weight of representative week $w$

### Decision variables

- $g_{i,y,w,t}$: Generation dispatch generator $i$ (MW)
- $f_{b,y,w,t}$: Power flow branch $b$ (MW)
- $sh_{n,y,w,t}$: Load shedding at node $n$ (MW)
- $c_{i,y,w,t}$: Curtailment generator $i$ (MW)
- $g_{s,y,w,t}^{ch}, g_{s,y,w,t}^{dis}$: Charge/discharge battery $s$ (MW)
- $soc_{s,y,w,t}$: State of charge battery $s$ (MWh)
- $soc_{s,y}^{max}$: Built storage capacity battery $s$ (MWh)
- $p_{i,y}^{max}$: Built capacity new generator $i$ (MW)
- $p_{b,y}^{max}$: Built capacity new branch $b$ (MW)

## Optimization Model

## Objective function

**Minimize:**
$$
\frac{1}{|Y|} \sum_{y \in Y} \left( AIC_y + OC_y \right)
$$


where

$$
OC_y = \sum_{w \in W}weight_w\left[
\sum_{i\in G}\sum_{t\in T_w}(MC_{i,y}+CO2_{i,y})g_{i,y,w,t} +
\sum_{s\in S}\sum_{t\in T_w}MC_{s,y}^{dis}g_{s,y,w,t}^{dis}\eta_s^{dis} +
\sum_{n\in N}\sum_{t\in T_w}sh_{n,y,w,t}VOLL_y +
\sum_{i\in G}\sum_{t\in T_w}c_{i,y,w,t}CC
\right]
$$

and

$$
AIC_y = \sum_{i\in G^{new}}AIC_{i,y}p_{i,y}^{max} + 
\sum_{b\in B^{new}}AIC_{b,y}p_{b,y}^{max} +
\sum_{s\in S^{new}}AIC_{s,y}soc_{s,y}^{max}
$$

---

### Constraints

1\. **Power balance**

$$
\sum_{i \in G_n}(g_{i,y,w,t}-c_{i,y,w,t}) + \sum_{b \in B_n^{in}} f_{b,y,w,t}(1 - l_{b}) - \sum_{b \in B_n^{out}} f_{b,y,w,t} - \sum_{s \in S_n}(g_{s,y,w,t}^{ch}-\eta_{s}^{dis}g_{s,y,w,t}^{dis}) + sh_{n,y,w,t} = D_{n,y,w,t}, \quad \forall n,y,w,t
$$

2a\. **Load shedding limit**

$$
sh_{n,y,w,t} \leq D_{n,y,w,t}, \quad \forall n,y,w,t
$$

2b\. **Curtailment limit**

$$
0 \leq c_{i,y,w,t} \leq g_{i,y,w,t}, \quad \forall i,y,w,t
$$

3a\. **Generator output limits (existing generators)**

$$
P_{i}^{\min} \leq g_{i,y,w,t} \leq P_{i}^{\max} cf_{i,y,w,t}, \quad \forall i \in G^{old},y,w,t
$$

3b\. **Generator output limits (new generators)**

$$
0 \leq g_{i,y,w,t} \leq cf_{i,y,w,t}\sum_{y'\in Y_y}p_{i,y'}^{max}, \quad \forall i \in G^{new},y,w,t
$$

3c\. **New generators' installed capacity limit**

$$
p_{i,y}^{max}\leq P_{i}^{\max}, \quad \forall i\in G^{new}, y
$$

4a\. **Branch flow limits (existing branches)**

$$
-P_{b}^{\max} \leq f_{b,y,w,t} \leq P_{b}^{\max}, \quad \forall b\in B^{old},y,w,t
$$

4b\. **Branch flow limits (new branches)**

$$
-\sum_{y'\in Y_y}p_{b,y'}^{max}\leq f_{b,y,w,t}\leq \sum_{y'\in Y_y}p_{b,y'}^{max},\quad \forall b\in B^{new},y,w,t
$$

4c\. **New branches' installed capacity limit**

$$
p_{b,y}^{max}\leq P_{b}^{\max}, \quad \forall b\in B^{new}, y
$$

5\. **Emissions restrictions**

$$
\sum_{i\in G}\sum_{y\in Y}\sum_{w\in W}\sum_{t\in T_w}E_{i,y}g_{i,y,w,t}\leq E_{limit}
$$

6a\. **Battery charging limit (old batteries)**

$$
P_{s}^{ch,\min}\leq g_{s,y,w,t}^{ch}\leq P_{s}^{ch,\max},\quad \forall s\in S^{old},y,w,t
$$

6b\. **Battery charging limit (new batteries)**

$$
0\leq g_{s,y,w,t}^{ch}\leq \frac{\sum_{y'\in Y_y}soc_{s,y'}^{max}}{batt_{hours}\cdot cdrate},\quad \forall s\in S^{new},y,w,t
$$

7a\. **Battery discharging limit (old batteries)**

$$
P_{s}^{dis,\min}\leq g_{s,y,w,t}^{dis}\leq P_{s}^{dis,\max},\quad \forall s\in S^{old},y,w,t
$$

7b\. **Battery discharging limit (new batteries)**

$$
0\leq g_{s,y,w,t}^{dis}\leq \frac{\sum_{y'\in Y_y}soc_{s,y'}^{max}}{batt_{hours}},\quad \forall s\in S^{new},y,w,t
$$

8\. **Battery state of charge limits**

$$
SOC_{s}^{\min}\cdot soc_{s,y}^{max}\leq soc_{s,y,w,t}\leq SOC_{s}^{\max}\cdot soc_{s,y}^{max},\quad \forall s,y,w,t
$$

9\. **Battery state of charge dynamics**

$$
soc_{s,y,w,t}=soc_{s,y,w,t-1}+\eta_{s}^{ch}g_{s,y,w,t}^{ch}-\frac{1}{\eta_{s}^{dis}}g_{s,y,w,t}^{dis},\quad \forall s,y,w,t\in T_w-\{0\}
$$

10a\. **Battery state of charge at time 0**

$$
soc_{s,y,w,0}=SOC_{s}^{\min}\cdot soc_{s,y}^{max},\quad \forall s,y,w
$$

10b\. **Battery state of charge at time T_w[-1]**

$$
soc_{s,y,w,T_w[-1]}=SOC_{s}^{\min}\cdot soc_{s,y}^{max},\quad \forall s,y,w
$$

11\. **Variable definitions**

All continuous variables are non-negative:

$$
g_{i,y,w,t},f_{b,y,w,t},sh_{n,y,w,t},c_{i,y,w,t},g_{s,y,w,t}^{ch},g_{s,y,w,t}^{dis},soc_{s,y,w,t},p_{i,y}^{max},p_{b,y}^{max},soc_{s,y}^{max}\geq0,\quad\forall i,b,n,s,y,w,t
$$

All binary variables (if present) are restricted to 0 or 1:

$$
\quad \forall i\in G^{new}, b\in B^{new}, s\in S^{new}, y\in Y


## Time-complexity

### 📊 Model Complexity Summary – Multi-Year with Representative Weeks

| Metric                     | Formula |
|----------------------------|---------|
| **Variables**              | $$ YWT(2g + b + n + 3s) + 2Y(g_n + b_n + s_n) $$ |
| **Constraints**            | $$ 2nYWT + 2gYWT + g_nY + bYWT + b_nY + 6sYWT + sYW + 1 $$ |
| **Parameters**             | $$ (n+g)YWT + (3g + s + g_n + b_n + s_n + b + 1)Y + 6g + 7s + 5b + W + 4 $$ |
| **Time-Coupled Constraints** | $$ sYW(T - 1) $$ |
